# 検索結果の上位からクエリを拡張する (擬似適合性フィードバック)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_search/query_expansion.ipynb)

NLP クックブック 18-10 ／ 検索

短いクエリでは語が足りず、関連文書に当たらない。最初の検索結果の上位に出てくる語をクエリに足して、もう一度検索したい。

In [ ]:
!pip install -q rank_bm25 fugashi unidic-lite

In [ ]:
import io, tarfile, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
from fugashi import Tagger
from rank_bm25 import BM25Okapi

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor().sample(1500, random_state=0).reset_index(drop=True)
tagger = Tagger()
tok = lambda s: [w.surface.lower() for w in tagger(s) if w.feature.pos1 in ("名詞", "動詞", "形容詞") and w.feature.pos2 not in ("非自立可能", "数詞") and len(w.surface) > 1]
corpus = [tok(t[:600]) for t in df["text"]]
bm25 = BM25Okapi(corpus)
doc_freq = Counter(t for d in corpus for t in set(d))

In [ ]:
def expand(query_terms, top_docs=5, n_terms=5):
    """上位文書に多く、コーパス全体では珍しい語 (TF × IDF) をクエリに足す"""
    first = np.argsort(-bm25.get_scores(query_terms))[:top_docs]
    tf = Counter(t for i in first for t in corpus[i])
    weight = {t: c * np.log(len(corpus) / doc_freq[t]) for t, c in tf.items() if t not in query_terms}
    return [t for t, _ in sorted(weight.items(), key=lambda x: -x[1])[:n_terms]]

for q in ["スマホ 電池", "サッカー 日本代表", "結婚 悩み"]:
    terms = tok(q)
    added = expand(terms)
    before = np.argsort(-bm25.get_scores(terms))[:3]
    after = np.argsort(-bm25.get_scores(terms + added))[:3]
    print(f"{q} + {added}")
    print("   拡張前:", [df['title'][i][:22] for i in before])
    print("   拡張後:", [df['title'][i][:22] for i in after])

In [ ]:
# 効果を測る: カテゴリが同じ記事を「関連あり」とみなして、上位 10 件の適合率を比べる
queries = {"スマホ アプリ": "smax", "映画 公開": "movie-enter", "試合 選手": "sports-watch", "恋愛 結婚": "dokujo-tsushin", "家電 売れ筋": "kaden-channel", "パソコン ソフト": "it-life-hack"}
rows = []
for q, label in queries.items():
    terms = tok(q)
    p = lambda ts: (df["label"].to_numpy()[np.argsort(-bm25.get_scores(ts))[:10]] == label).mean()
    rows.append({"クエリ": q, "拡張前 P@10": p(terms), "拡張後 P@10": p(terms + expand(terms)), "足した語": " ".join(expand(terms))})
print(pd.DataFrame(rows).to_string(index=False))

- 「最初の検索の上位はだいたい関連文書だろう」と仮定して、そこから特徴的な語を拾う。人の判断を使わないので「擬似」適合性フィードバックと呼ぶ。
- 上位が外れていると、無関係な語が足されて検索がさらにずれる (クエリドリフト)。足す語数と使う文書数は控えめにする。
- 元のクエリの語を、足した語より重くするのが普通 (ここでは同じ重み)。RM3 と呼ばれる方式が古典的な標準。
- 同義語辞書 (WordNet) による拡張や、LLM にクエリを言い換えさせる方法もある。埋め込み検索は、同じ効果を内部で実現していると言える。